In [ ]:
# Загрузка и импорт необходимых библиотек
!pip install -q pandas numpy
!pip install ucimlrepo -q

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.impute import SimpleImputer

В работе используется датасет Communities and Crime, содержащий социально-экономические и демографические характеристики различных сообществ США, а также информацию об уровне преступности.
В датасете содержится 1994 объекта и 127 исходных признаков. Признаки описывают различные характеристики территорий: состав населения, уровень доходов, занятость, семейное положение и другие показатели.
Целевая переменная — ViolentCrimesPerPop, характеризующая уровень насильственных преступлений относительно численности населения.

In [ ]:
data = fetch_ucirepo(id=183)

X = data.data.features
y = data.data.targets

print("Размер X:", X.shape)
print("Размер y:", y.shape)

display(X.head())
display(y.head())

Размер X: (1994, 127)
Размер y: (1994, 1)


,state,county,community,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,...,PolicAveOTWorked,LandArea,PopDens,PctUsePubTrans,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,LemasPctOfficDrugUn,PolicBudgPerPop
0,8,?,?,Lakewoodcity,1,0.19,0.33,0.02,0.90,0.12,...,0.29,0.12,0.26,0.20,0.06,0.04,0.9,0.5,0.32,0.14
1,53,?,?,Tukwilacity,1,0.00,0.16,0.12,0.74,0.45,...,?,0.02,0.12,0.45,?,?,?,?,0.00,?
2,24,?,?,Aberdeentown,1,0.00,0.42,0.49,0.56,0.17,...,?,0.01,0.21,0.02,?,?,?,?,0.00,?
3,34,5,81440,Willingborotownship,1,0.04,0.77,1.00,0.08,0.12,...,?,0.02,0.39,0.28,?,?,?,?,0.00,?
4,42,95,6096,Bethlehemtownship,1,0.01,0.55,0.02,0.95,0.09,...,?,0.04,0.09,0.02,?,?,?,?,0.00,?


,ViolentCrimesPerPop
0,0.20
1,0.67
2,0.43
3,0.12
4,0.03


In [ ]:
print("Размерность признаков:", X.shape)
print("Размерность целевой переменной:", y.shape)

print("\nЦелевая переменная:")
print(y.columns)

print("\nТипы данных:")
print(X.dtypes.value_counts())

print("\nКоличество пропущенных значений:")
print(X.isnull().sum().sum())

print("\nКоличество дубликатов:")
print(X.duplicated().sum())

display(X.describe())

Размерность признаков: (1994, 127)
Размерность целевой переменной: (1994, 1)

Целевая переменная:
Index(['ViolentCrimesPerPop'], dtype='object')

Типы данных:
float64    99
object     26
int64       2
Name: count, dtype: int64

Количество пропущенных значений:
0

Количество дубликатов:
0


,state,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,agePct12t29,...,NumStreet,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans,LemasPctOfficDrugUn
count,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,...,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000,1994.000000
mean,28.683551,5.493982,0.057593,0.463395,0.179629,0.753716,0.153681,0.144022,0.424218,0.493867,...,0.022778,0.215552,0.608892,0.535050,0.626424,0.651530,0.065231,0.232854,0.161685,0.094052
std,16.397553,2.873694,0.126906,0.163717,0.253442,0.244039,0.208877,0.232492,0.155196,0.143564,...,0.100400,0.231134,0.204329,0.181352,0.200521,0.198221,0.109459,0.203092,0.229055,0.240328
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.000000,3.000000,0.010000,0.350000,0.020000,0.630000,0.040000,0.010000,0.340000,0.410000,...,0.000000,0.060000,0.470000,0.420000,0.520000,0.560000,0.020000,0.100000,0.020000,0.000000
50%,34.000000,5.000000,0.020000,0.440000,0.060000,0.850000,0.070000,0.040000,0.400000,0.480000,...,0.000000,0.130000,0.630000,0.540000,0.670000,0.700000,0.040000,0.170000,0.070000,0.000000
75%,42.000000,8.000000,0.050000,0.540000,0.230000,0.940000,0.170000,0.160000,0.470000,0.540000,...,0.000000,0.280000,0.777500,0.660000,0.770000,0.790000,0.070000,0.280000,0.190000,0.000000
max,56.000000,10.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
# Признаки типа object
object_columns = X.select_dtypes(include="object").columns

print("Количество признаков типа object:", len(object_columns))
print("\nНазвания:")
print(object_columns.tolist())

print("\nПримеры значений:")
display(X[object_columns].head())

Количество признаков типа object: 26

Названия:
['county', 'community', 'communityname', 'OtherPerCap', 'LemasSwornFT', 'LemasSwFTPerPop', 'LemasSwFTFieldOps', 'LemasSwFTFieldPerPop', 'LemasTotalReq', 'LemasTotReqPerPop', 'PolicReqPerOffic', 'PolicPerPop', 'RacialMatchCommPol', 'PctPolicWhite', 'PctPolicBlack', 'PctPolicHisp', 'PctPolicAsian', 'PctPolicMinor', 'OfficAssgnDrugUnits', 'NumKindsDrugsSeiz', 'PolicAveOTWorked', 'PolicCars', 'PolicOperBudg', 'LemasPctPolicOnPatr', 'LemasGangUnitDeploy', 'PolicBudgPerPop']

Примеры значений:


,county,community,communityname,OtherPerCap,LemasSwornFT,LemasSwFTPerPop,LemasSwFTFieldOps,LemasSwFTFieldPerPop,LemasTotalReq,LemasTotReqPerPop,...,PctPolicAsian,PctPolicMinor,OfficAssgnDrugUnits,NumKindsDrugsSeiz,PolicAveOTWorked,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,PolicBudgPerPop
0,?,?,Lakewoodcity,0.36,0.03,0.13,0.96,0.17,0.06,0.18,...,0.1,0.07,0.02,0.57,0.29,0.06,0.04,0.9,0.5,0.14
1,?,?,Tukwilacity,0.22,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,?
2,?,?,Aberdeentown,0.28,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,?
3,5,81440,Willingborotownship,0.36,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,?
4,95,6096,Bethlehemtownship,0.51,?,?,?,?,?,?,...,?,?,?,?,?,?,?,?,?,?


In [ ]:
for col in object_columns:
    print(
        col,
        "— уникальных значений:",
        X[col].nunique(),
        "— примеры:",
        X[col].dropna().unique()[:5]
    )

county — уникальных значений: 109 — примеры: ['?' '5' '95' '7' '21']
community — уникальных значений: 800 — примеры: ['?' '81440' '6096' '41500' '44105']
communityname — уникальных значений: 1828 — примеры: ['Lakewoodcity' 'Tukwilacity' 'Aberdeentown' 'Willingborotownship'
 'Bethlehemtownship']
OtherPerCap — уникальных значений: 98 — примеры: ['0.36' '0.22' '0.28' '0.51' '0.48']
LemasSwornFT — уникальных значений: 39 — примеры: ['0.03' '?' '0.06' '0.25' '0.01']
LemasSwFTPerPop — уникальных значений: 53 — примеры: ['0.13' '?' '0.3' '0.08' '0.16']
LemasSwFTFieldOps — уникальных значений: 35 — примеры: ['0.96' '?' '0.93' '0.75' '0.98']
LemasSwFTFieldPerPop — уникальных значений: 56 — примеры: ['0.17' '?' '0.38' '0.12' '0.19']
LemasTotalReq — уникальных значений: 45 — примеры: ['0.06' '?' '0.05' '0.35' '0.03']
LemasTotReqPerPop — уникальных значений: 60 — примеры: ['0.18' '?' '0.21' '0.11' '0.19']
PolicReqPerOffic — уникальных значений: 76 — примеры: ['0.44' '?' '0.23' '0.35' '0.38']
Polic

In [ ]:
# Удаляем идентификационные признаки
X = X.drop(columns=["county", "community", "communityname"])

# Заменяем '?' на пропуски
X = X.replace("?", np.nan)

# Преобразуем все признаки в числовой тип
X = X.apply(pd.to_numeric)

print("Размерность после удаления идентификаторов:", X.shape)
print("Количество пропущенных значений:", X.isnull().sum().sum())
print("\nТипы данных:")
print(X.dtypes.value_counts())

Размерность после удаления идентификаторов: (1994, 124)
Количество пропущенных значений: 36851

Типы данных:
float64    122
int64        2
Name: count, dtype: int64


In [ ]:
# Сначала 70% train и 30% для validation + test
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Оставшиеся 30% делим пополам:
# 15% validation и 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1395, 124)
Validation: (299, 124)
Test: (300, 124)


In [ ]:
# Заполнение пропущенных значений медианой
imputer = SimpleImputer(strategy="median")

# Медианы вычисляем только по обучающей выборке
X_train_imputed = imputer.fit_transform(X_train)

# Для validation и test используем те же медианы
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("Пропусков в train:", np.isnan(X_train_imputed).sum())
print("Пропусков в validation:", np.isnan(X_val_imputed).sum())
print("Пропусков в test:", np.isnan(X_test_imputed).sum())

Пропусков в train: 0
Пропусков в validation: 0
Пропусков в test: 0


In [ ]:
# Стандартизация признаков
scaler = StandardScaler()

# Стандартизируем данные, в которых уже заполнены пропуски
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Размерность обучающей выборки:", X_train_scaled.shape)
print("Размерность валидационной выборки:", X_val_scaled.shape)
print("Размерность тестовой выборки:", X_test_scaled.shape)

print("\nСредние значения train после стандартизации:")
print(np.mean(X_train_scaled, axis=0))

print("\nСтандартные отклонения train после стандартизации:")
print(np.std(X_train_scaled, axis=0))

print("\nNaN после стандартизации:", np.isnan(X_train_scaled).sum())

Размерность обучающей выборки: (1395, 124)
Размерность валидационной выборки: (299, 124)
Размерность тестовой выборки: (300, 124)

Средние значения train после стандартизации:
[-1.27337408e-17  9.67764300e-17  3.18343520e-17  1.68085378e-16
 -6.36687039e-19  2.36847579e-16 -5.34817113e-17  1.06963423e-16
  6.87622002e-17 -2.80142297e-16 -1.73178875e-16 -3.04336405e-16
 -3.82012224e-17  1.06963423e-16 -1.69358752e-16 -1.78272371e-16
  1.98646356e-16  3.32350634e-16  3.57818116e-16  0.00000000e+00
 -3.46357749e-16  1.27337408e-17  9.67764300e-17 -2.52128068e-16
  2.35574205e-16 -2.20293716e-16 -8.14959410e-17 -1.91006112e-17
 -1.71905501e-16  0.00000000e+00  2.13926845e-16  3.05609779e-17
 -2.82689045e-16  8.73852961e-17 -9.55030559e-17 -2.47034571e-16
 -7.25823225e-17 -1.65538630e-16  1.92279486e-16  1.29884156e-16
 -1.89732738e-16  4.07479705e-17 -1.37524400e-16  1.56625012e-16
 -3.20890268e-16 -3.82012224e-16  1.83365867e-16 -2.34300830e-16
 -3.25983764e-16 -6.16313054e-16  2.77595549

In [ ]:
# Ковариационная матрица
cov_matrix = np.cov(X_train_scaled, rowvar=False)

print("Размер ковариационной матрицы:", cov_matrix.shape)
print(cov_matrix)

Размер ковариационной матрицы: (124, 124)
[[ 1.00071736 -0.0011025  -0.07029838 ... -0.05280766 -0.0602865
  -0.06869005]
 [-0.0011025   1.00071736  0.00507665 ... -0.00145842 -0.02904349
   0.029806  ]
 [-0.07029838  0.00507665  1.00071736 ...  0.00974884  0.44940219
   0.09781055]
 ...
 [-0.05280766 -0.00145842  0.00974884 ...  1.00071736 -0.11142934
  -0.0537972 ]
 [-0.0602865  -0.02904349  0.44940219 ... -0.11142934  1.00071736
   0.21971085]
 [-0.06869005  0.029806    0.09781055 ... -0.0537972   0.21971085
   1.00071736]]


In [ ]:
# Собственные значения и собственные векторы
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Сортировка по убыванию собственных значений
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Собственные значения:")
print(eigenvalues)

Собственные значения:
[2.58463099e+01 1.83553484e+01 1.16076233e+01 8.24956542e+00
 7.02836970e+00 4.53621553e+00 4.07841503e+00 3.71338978e+00
 3.14405947e+00 2.22677916e+00 2.03880362e+00 1.71988276e+00
 1.69612212e+00 1.57303472e+00 1.46040146e+00 1.37552081e+00
 1.35383426e+00 1.11292892e+00 1.06389524e+00 1.02810006e+00
 9.84037131e-01 9.66221335e-01 9.07150094e-01 8.91890213e-01
 8.19892680e-01 7.77662840e-01 7.07175880e-01 6.88479789e-01
 6.70191788e-01 6.40459148e-01 6.19004088e-01 5.95845892e-01
 5.45253175e-01 5.27435069e-01 5.23023414e-01 4.95934041e-01
 4.75318781e-01 4.55022702e-01 4.27543820e-01 3.99704459e-01
 3.96223813e-01 3.64719502e-01 3.50796827e-01 3.39125310e-01
 3.35741070e-01 3.08645046e-01 3.04523033e-01 2.72965197e-01
 2.53989785e-01 2.47158905e-01 2.34952670e-01 2.23844374e-01
 2.08896510e-01 2.04729236e-01 1.95266155e-01 1.89581876e-01
 1.84877053e-01 1.76282844e-01 1.70458447e-01 1.57529898e-01
 1.43128278e-01 1.37009547e-01 1.25749381e-01 1.24976422e-01
 1

In [ ]:
explained_variance_ratio = eigenvalues / eigenvalues.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)

for i in range(len(eigenvalues)):
    print(
        f"Компонента {i + 1}: "
        f"{explained_variance_ratio[i] * 100:.2f}% "
        f"(накопленная: {cumulative_variance[i] * 100:.2f}%)"
    )

Компонента 1: 20.83% (накопленная: 20.83%)
Компонента 2: 14.79% (накопленная: 35.62%)
Компонента 3: 9.35% (накопленная: 44.98%)
Компонента 4: 6.65% (накопленная: 51.62%)
Компонента 5: 5.66% (накопленная: 57.29%)
Компонента 6: 3.66% (накопленная: 60.94%)
Компонента 7: 3.29% (накопленная: 64.23%)
Компонента 8: 2.99% (накопленная: 67.22%)
Компонента 9: 2.53% (накопленная: 69.76%)
Компонента 10: 1.79% (накопленная: 71.55%)
Компонента 11: 1.64% (накопленная: 73.19%)
Компонента 12: 1.39% (накопленная: 74.58%)
Компонента 13: 1.37% (накопленная: 75.95%)
Компонента 14: 1.27% (накопленная: 77.21%)
Компонента 15: 1.18% (накопленная: 78.39%)
Компонента 16: 1.11% (накопленная: 79.50%)
Компонента 17: 1.09% (накопленная: 80.59%)
Компонента 18: 0.90% (накопленная: 81.49%)
Компонента 19: 0.86% (накопленная: 82.34%)
Компонента 20: 0.83% (накопленная: 83.17%)
Компонента 21: 0.79% (накопленная: 83.97%)
Компонента 22: 0.78% (накопленная: 84.74%)
Компонента 23: 0.73% (накопленная: 85.48%)
Компонента 24: 0.7

Для снижения размерности были использованы методы PCA и Truncated SVD. После предварительной обработки исходное признаковое пространство состояло из 124 признаков. Количество компонент выбиралось на основе накопленной объяснённой дисперсии. Было выбрано 31 компоненту, которые сохраняют около 90.16% дисперсии исходных стандартизированных данных. Таким образом, размерность признакового пространства была сокращена с 124 до 31.
Для PCA также была выполнена собственная реализация на основе ковариационной матрицы, собственных значений и собственных векторов. Результаты собственной реализации были сопоставлены с PCA из библиотеки sklearn. Полученная доля объяснённой дисперсии совпала, что подтверждает корректность реализации.
Truncated SVD с 31 компонентой также сохранил около 90.16% дисперсии. В данном случае результаты PCA и Truncated SVD практически совпадают, поскольку перед снижением размерности признаки были стандартизированы и, следовательно, центрированы.

### Математический принцип PCA

Метод главных компонент (PCA) используется для уменьшения размерности данных с сохранением как можно большей части их дисперсии.

Сначала данные центрируются:

$$
X_c = X - \mu,
$$

где $X$ — исходная матрица признаков, $\mu$ — вектор средних значений признаков, $X_c$ — центрированная матрица.

Затем вычисляется ковариационная матрица:

$$
C = \frac{1}{n-1}X_c^T X_c,
$$

где $n$ — количество объектов.

Для ковариационной матрицы находятся собственные значения $\lambda_i$ и соответствующие им собственные векторы $v_i$:

$$
Cv_i = \lambda_i v_i.
$$

Собственные векторы задают направления главных компонент, а собственные значения характеризуют количество дисперсии, сохраняемой каждой компонентой.

Компоненты сортируются по убыванию собственных значений, после чего выбираются первые $k$ компонент. Преобразование данных выполняется по формуле:

$$
Z = X_c W_k,
$$

где $W_k$ — матрица из $k$ выбранных собственных векторов, а $Z$ — представление данных в пространстве меньшей размерности.

В данной работе количество компонент выбирается таким образом, чтобы сохранить не менее 90 % исходной дисперсии данных.

In [ ]:
# Количество выбранных компонент
n_components = 31

# Выбираем собственные векторы,
# соответствующие n_components наибольшим собственным значениям
W = eigenvectors[:, :n_components]

# Проекция обучающих и тестовых данных
X_train_pca_manual = X_train_scaled @ W
X_val_pca_manual = X_val_scaled @ W
X_test_pca_manual = X_test_scaled @ W

print("Исходная размерность:", X_train_scaled.shape)
print("Размерность после PCA:", X_train_pca_manual.shape)

print(
    "Сохранённая дисперсия:",
    f"{explained_variance_ratio[:n_components].sum() * 100:.2f}%"
)

Исходная размерность: (1395, 124)
Размерность после PCA: (1395, 31)
Сохранённая дисперсия: 90.16%


In [ ]:
# PCA из sklearn
pca = PCA(n_components = 31)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Размерность train после PCA:", X_train_pca.shape)
print("Размерность test после PCA:", X_test_pca.shape)

print("\nОбъяснённая дисперсия по компонентам:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"Компонента {i + 1}: {ratio * 100:.2f}%")

print(
    "\nСуммарная объяснённая дисперсия:",
    f"{pca.explained_variance_ratio_.sum() * 100:.2f}%"
)

Размерность train после PCA: (1395, 31)
Размерность test после PCA: (300, 31)

Объяснённая дисперсия по компонентам:
Компонента 1: 20.83%
Компонента 2: 14.79%
Компонента 3: 9.35%
Компонента 4: 6.65%
Компонента 5: 5.66%
Компонента 6: 3.66%
Компонента 7: 3.29%
Компонента 8: 2.99%
Компонента 9: 2.53%
Компонента 10: 1.79%
Компонента 11: 1.64%
Компонента 12: 1.39%
Компонента 13: 1.37%
Компонента 14: 1.27%
Компонента 15: 1.18%
Компонента 16: 1.11%
Компонента 17: 1.09%
Компонента 18: 0.90%
Компонента 19: 0.86%
Компонента 20: 0.83%
Компонента 21: 0.79%
Компонента 22: 0.78%
Компонента 23: 0.73%
Компонента 24: 0.72%
Компонента 25: 0.66%
Компонента 26: 0.63%
Компонента 27: 0.57%
Компонента 28: 0.55%
Компонента 29: 0.54%
Компонента 30: 0.52%
Компонента 31: 0.50%

Суммарная объяснённая дисперсия: 90.16%


In [ ]:
# Доля объяснённой дисперсии собственного PCA
manual_ratio = eigenvalues[:n_components] / eigenvalues.sum()

# Доля объяснённой дисперсии sklearn PCA
sklearn_ratio = pca.explained_variance_ratio_

print("Сохранённая дисперсия:")
print(f"Собственный PCA: {manual_ratio.sum():.6f}")
print(f"sklearn PCA:     {sklearn_ratio.sum():.6f}")

print(
    "\nМаксимальная разница explained variance ratio:",
    np.max(np.abs(manual_ratio - sklearn_ratio))
)

Сохранённая дисперсия:
Собственный PCA: 0.901617
sklearn PCA:     0.901617

Максимальная разница explained variance ratio: 1.1102230246251565e-16


In [ ]:
X_manual_aligned = X_train_pca_manual.copy()

for i in range(n_components):
    corr = np.corrcoef(
        X_manual_aligned[:, i],
        X_train_pca[:, i]
    )[0, 1]

    if corr < 0:
        X_manual_aligned[:, i] *= -1

difference = np.max(
    np.abs(X_manual_aligned - X_train_pca)
)

print("Максимальная разница после выравнивания знаков:", difference)

Максимальная разница после выравнивания знаков: 3.979039320256561e-13


PCA использовался для снижения размерности признакового пространства. На обучающей выборке была построена ковариационная матрица, для которой вычислены собственные значения и собственные векторы. Собственные векторы определяют направления главных компонент, а собственные значения характеризуют объясняемую ими дисперсию. Были выбраны первые 31 компонента, которые сохраняют около 90,16% дисперсии. В результате размерность была уменьшена со 124 до 31 признака.

Собственная реализация PCA и реализация из библиотеки sklearn сохраняют практически одинаковую долю дисперсии. Полученные проекции также близки после учёта возможного различия знаков главных компонент. Различие знаков не является ошибкой, поскольку собственный вектор может быть умножен на −1 без изменения соответствующего ему направления. Небольшие численные различия могут быть связаны с особенностями используемых алгоритмов вычислений и точностью операций с числами с плавающей точкой.

In [ ]:
svd = TruncatedSVD(
    n_components=31,
    random_state=42
)

X_train_svd = svd.fit_transform(X_train_scaled)
X_val_svd = svd.transform(X_val_scaled)
X_test_svd = svd.transform(X_test_scaled)

print("Размерность train после SVD:", X_train_svd.shape)
print("Размерность test после SVD:", X_test_svd.shape)

print("\nОбъяснённая дисперсия по компонентам:")
for i, ratio in enumerate(svd.explained_variance_ratio_):
    print(f"Компонента {i + 1}: {ratio * 100:.2f}%")

print(
    "\nСуммарная объяснённая дисперсия:",
    f"{svd.explained_variance_ratio_.sum() * 100:.2f}%"
)

Размерность train после SVD: (1395, 31)
Размерность test после SVD: (300, 31)

Объяснённая дисперсия по компонентам:
Компонента 1: 20.83%
Компонента 2: 14.79%
Компонента 3: 9.35%
Компонента 4: 6.65%
Компонента 5: 5.66%
Компонента 6: 3.66%
Компонента 7: 3.29%
Компонента 8: 2.99%
Компонента 9: 2.53%
Компонента 10: 1.79%
Компонента 11: 1.64%
Компонента 12: 1.39%
Компонента 13: 1.37%
Компонента 14: 1.27%
Компонента 15: 1.18%
Компонента 16: 1.11%
Компонента 17: 1.09%
Компонента 18: 0.90%
Компонента 19: 0.86%
Компонента 20: 0.83%
Компонента 21: 0.79%
Компонента 22: 0.78%
Компонента 23: 0.73%
Компонента 24: 0.72%
Компонента 25: 0.66%
Компонента 26: 0.63%
Компонента 27: 0.57%
Компонента 28: 0.55%
Компонента 29: 0.54%
Компонента 30: 0.52%
Компонента 31: 0.50%

Суммарная объяснённая дисперсия: 90.16%


### Линейная регрессия

Линейная регрессия описывает зависимость целевой переменной от признаков в виде:

$$
\hat{y} = Xw,
$$

где $X$ — матрица признаков, $w$ — вектор коэффициентов модели, а $\hat{y}$ — предсказанные значения.

Для определения коэффициентов используется метод наименьших квадратов, минимизирующий сумму квадратов ошибок:

$$
\min_w ||Xw-y||_2^2.
$$

Решение задачи имеет вид:

$$
w^*=(X^TX)^{-1}X^Ty,
$$

где $y$ — вектор истинных значений целевой переменной.

В работе данная формула реализуется самостоятельно, после чего полученные коэффициенты, предсказания и метрики сравниваются с реализацией LinearRegression из библиотеки sklearn.

In [ ]:
# Добавляем столбец единиц для свободного коэффициента
X_train_bias = np.column_stack([
    np.ones(X_train_pca.shape[0]),
    X_train_pca
])

X_test_bias = np.column_stack([
    np.ones(X_test_pca.shape[0]),
    X_test_pca
])

# Формула метода наименьших квадратов
w_manual = (
    np.linalg.inv(X_train_bias.T @ X_train_bias)
    @ X_train_bias.T
    @ y_train.to_numpy()
)

# Предсказания
y_pred_manual = X_test_bias @ w_manual

print("Количество коэффициентов:", len(w_manual))
print("Свободный коэффициент:", w_manual[0])
print("Остальные коэффициенты:")
print(w_manual[1:])

print("\nПервые 10 предсказаний:")
print(y_pred_manual[:10])

print("\nПервые 10 настоящих значений:")
print(y_test.to_numpy()[:10])

Количество коэффициентов: 32
Свободный коэффициент: [0.24506093]
Остальные коэффициенты:
[[-0.03138834]
 [ 0.01182554]
 [-0.00908845]
 [-0.00149151]
 [ 0.01457931]
 [-0.00110207]
 [ 0.00294188]
 [ 0.0327027 ]
 [-0.02371545]
 [ 0.00294037]
 [ 0.00480763]
 [ 0.00254189]
 [-0.00557187]
 [-0.0015351 ]
 [ 0.00015417]
 [-0.0022912 ]
 [-0.01571083]
 [ 0.00258333]
 [-0.01197376]
 [ 0.00341729]
 [ 0.00726948]
 [ 0.00596646]
 [-0.00691557]
 [ 0.00228836]
 [ 0.00335873]
 [-0.00324074]
 [-0.00285074]
 [ 0.00914065]
 [ 0.00932089]
 [-0.00329893]
 [ 0.00299613]]

Первые 10 предсказаний:
[[0.07175587]
 [0.26252857]
 [0.30126419]
 [0.07571427]
 [0.22929538]
 [0.40023201]
 [0.49046885]
 [0.60590415]
 [0.15811901]
 [0.08065614]]

Первые 10 настоящих значений:
[[0.04]
 [0.15]
 [0.33]
 [0.06]
 [0.1 ]
 [0.54]
 [0.63]
 [0.86]
 [0.15]
 [0.04]]


In [ ]:
# Линейная регрессия sklearn
lr = LinearRegression()

lr.fit(X_train_pca, y_train)

# Предсказания
y_pred_sklearn = lr.predict(X_test_pca)

print("Свободный коэффициент:")
print("Вручную:", w_manual[0])
print("sklearn:", lr.intercept_)

print("\nКоэффициенты:")
print("Вручную:", w_manual[1:])
print("sklearn:", lr.coef_)

print("\nМаксимальная разница между предсказаниями:")
print(np.max(np.abs(y_pred_manual - y_pred_sklearn)))

print("\nПервые 10 предсказаний sklearn:")
print(y_pred_sklearn[:10])

Свободный коэффициент:
Вручную: [0.24506093]
sklearn: [0.24506093]

Коэффициенты:
Вручную: [[-0.03138834]
 [ 0.01182554]
 [-0.00908845]
 [-0.00149151]
 [ 0.01457931]
 [-0.00110207]
 [ 0.00294188]
 [ 0.0327027 ]
 [-0.02371545]
 [ 0.00294037]
 [ 0.00480763]
 [ 0.00254189]
 [-0.00557187]
 [-0.0015351 ]
 [ 0.00015417]
 [-0.0022912 ]
 [-0.01571083]
 [ 0.00258333]
 [-0.01197376]
 [ 0.00341729]
 [ 0.00726948]
 [ 0.00596646]
 [-0.00691557]
 [ 0.00228836]
 [ 0.00335873]
 [-0.00324074]
 [-0.00285074]
 [ 0.00914065]
 [ 0.00932089]
 [-0.00329893]
 [ 0.00299613]]
sklearn: [[-0.03138834  0.01182554 -0.00908845 -0.00149151  0.01457931 -0.00110207
   0.00294188  0.0327027  -0.02371545  0.00294037  0.00480763  0.00254189
  -0.00557187 -0.0015351   0.00015417 -0.0022912  -0.01571083  0.00258333
  -0.01197376  0.00341729  0.00726948  0.00596646 -0.00691557  0.00228836
   0.00335873 -0.00324074 -0.00285074  0.00914065  0.00932089 -0.00329893
   0.00299613]]

Максимальная разница между предсказаниями:
1.16

In [ ]:
# Метрики собственной реализации
mse_manual = mean_squared_error(y_test, y_pred_manual)
rmse_manual = np.sqrt(mse_manual)
r2_manual = r2_score(y_test, y_pred_manual)

# Метрики sklearn
mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
rmse_sklearn = np.sqrt(mse_sklearn)
r2_sklearn = r2_score(y_test, y_pred_sklearn)

print("Собственная реализация:")
print(f"MSE  = {mse_manual:.6f}")
print(f"RMSE = {rmse_manual:.6f}")
print(f"R²   = {r2_manual:.6f}")

print("\nsklearn LinearRegression:")
print(f"MSE  = {mse_sklearn:.6f}")
print(f"RMSE = {rmse_sklearn:.6f}")
print(f"R²   = {r2_sklearn:.6f}")

Собственная реализация:
MSE  = 0.017524
RMSE = 0.132377
R²   = 0.631797

sklearn LinearRegression:
MSE  = 0.017524
RMSE = 0.132377
R²   = 0.631797


Полученные значения MSE, RMSE и \(R^2\) для собственной реализации линейной регрессии и LinearRegression из библиотеки sklearn практически совпадают. Аналогично, различия между коэффициентами и предсказаниями находятся на уровне вычислительной погрешности. Это подтверждает корректность собственной реализации метода наименьших квадратов.

### Регуляризация

Регуляризация используется для уменьшения переобучения модели за счёт добавления штрафа за величину коэффициентов.

В Ridge-регрессии минимизируется функция:

$$
\min_w \left( ||Xw-y||_2^2 + \alpha ||w||_2^2 \right),
$$

где $X$ — матрица признаков, $y$ — истинные значения целевой переменной, $w$ — коэффициенты модели, а $\alpha$ — коэффициент регуляризации.

При увеличении $\alpha$ штраф за большие значения коэффициентов возрастает, поэтому коэффициенты модели уменьшаются. При слишком большом значении $\alpha$ модель может стать слишком простой и качество предсказаний снизится.

Для исследования влияния силы регуляризации модели обучаются при различных значениях $\alpha$. Выбор параметра выполняется по результатам на валидационной выборке.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

ridge_results = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_pca, y_train.values.ravel())

    y_pred = ridge.predict(X_val_pca)

    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_pred)

    ridge_results.append({
        "alpha": alpha,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print(
        f"alpha={alpha:<7} "
        f"MSE={mse:.6f} "
        f"RMSE={rmse:.6f} "
        f"R²={r2:.6f}"
    )

alpha=0.001   MSE=0.016727 RMSE=0.129333 R²=0.652689
alpha=0.01    MSE=0.016727 RMSE=0.129333 R²=0.652689
alpha=0.1     MSE=0.016727 RMSE=0.129333 R²=0.652689
alpha=1       MSE=0.016727 RMSE=0.129333 R²=0.652690
alpha=10      MSE=0.016726 RMSE=0.129331 R²=0.652700
alpha=100     MSE=0.016727 RMSE=0.129332 R²=0.652691
alpha=1000    MSE=0.016982 RMSE=0.130313 R²=0.647403


In [ ]:
ridge_results_df = pd.DataFrame(ridge_results)

best_idx = ridge_results_df["RMSE"].idxmin()
best_alpha_ridge = ridge_results_df.loc[best_idx, "alpha"]

print("Лучшее значение alpha:", best_alpha_ridge)

ridge_results_df

Лучшее значение alpha: 10.0


,alpha,MSE,RMSE,R2
0,0.001,0.016727,0.129333,0.652689
1,0.010,0.016727,0.129333,0.652689
2,0.100,0.016727,0.129333,0.652689
3,1.000,0.016727,0.129333,0.652690
4,10.000,0.016726,0.129331,0.652700
5,100.000,0.016727,0.129332,0.652691
6,1000.000,0.016982,0.130313,0.647403


По результатам эксперимента значения метрик при $\alpha$ от 0.001 до 100 изменяются незначительно. Наилучший результат на валидационной выборке получен при $\alpha = 10$: MSE = 0.016726, RMSE = 0.129331 и $R^2 = 0.652700$.

При увеличении $\alpha$ до 1000 качество модели ухудшается: RMSE возрастает до 0.130313, а $R^2$ снижается до 0.647403. Это связано с усилением регуляризации, которая сильнее ограничивает значения коэффициентов модели и при слишком больших значениях $\alpha$ может приводить к недообучению.

Для итоговой оценки Ridge-регрессии выбрано значение $\alpha = 10$.

In [ ]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1]

lasso_results = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_pca, y_train.values.ravel())

    y_val_pred = lasso.predict(X_val_pca)

    mse = mean_squared_error(y_val, y_val_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_val_pred)

    nonzero = np.sum(lasso.coef_ != 0)

    lasso_results.append({
        "alpha": alpha,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "nonzero": nonzero
    })

    print(
        f"alpha = {alpha}: "
        f"MSE = {mse:.6f}, "
        f"RMSE = {rmse:.6f}, "
        f"R² = {r2:.6f}, "
        f"ненулевых коэффициентов = {nonzero}"
    )

best_lasso = min(lasso_results, key=lambda x: x["RMSE"])
best_alpha_lasso = best_lasso["alpha"]

print("\nЛучший alpha:", best_alpha_lasso)
print("Лучший RMSE:", best_lasso["RMSE"])

alpha = 0.0001: MSE = 0.016729, RMSE = 0.129343, R² = 0.652637, ненулевых коэффициентов = 31
alpha = 0.001: MSE = 0.016774, RMSE = 0.129513, R² = 0.651719, ненулевых коэффициентов = 30
alpha = 0.01: MSE = 0.017674, RMSE = 0.132943, R² = 0.633029, ненулевых коэффициентов = 10
alpha = 0.1: MSE = 0.024082, RMSE = 0.155183, R² = 0.499975, ненулевых коэффициентов = 5
alpha = 1: MSE = 0.048805, RMSE = 0.220918, R² = -0.013356, ненулевых коэффициентов = 0

Лучший alpha: 0.0001
Лучший RMSE: 0.12934256248430637


In [ ]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1]

for alpha in alphas:
    elastic = ElasticNet(
        alpha=alpha,
        l1_ratio=0.5,
        max_iter=10000
    )

    elastic.fit(X_train_pca, y_train)
    y_pred_elastic = elastic.predict(X_val_pca)

    mse = mean_squared_error(y_val, y_pred_elastic)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_pred_elastic)

    print(f"alpha = {alpha}")
    print(f"MSE  = {mse:.4f}")
    print(f"RMSE = {rmse:.4f}")
    print(f"R²   = {r2:.4f}")
    print(
        "Ненулевых коэффициентов =",
        np.sum(elastic.coef_ != 0)
    )
    print()

alpha = 0.0001
MSE  = 0.0167
RMSE = 0.1293
R²   = 0.6527
Ненулевых коэффициентов = 31

alpha = 0.001
MSE  = 0.0167
RMSE = 0.1294
R²   = 0.6523
Ненулевых коэффициентов = 30

alpha = 0.01
MSE  = 0.0172
RMSE = 0.1313
R²   = 0.6421
Ненулевых коэффициентов = 18

alpha = 0.1
MSE  = 0.0197
RMSE = 0.1402
R²   = 0.5918
Ненулевых коэффициентов = 6

alpha = 1
MSE  = 0.0344
RMSE = 0.1855
R²   = 0.2852
Ненулевых коэффициентов = 1



In [ ]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1]
l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9]

elastic_results = []

for l1_ratio in l1_ratios:
    for alpha in alphas:

        elastic = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=10000
        )

        elastic.fit(
            X_train_pca,
            y_train.values.ravel()
        )

        y_val_pred = elastic.predict(X_val_pca)

        mse = mean_squared_error(y_val, y_val_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_val, y_val_pred)

        nonzero = np.sum(elastic.coef_ != 0)

        elastic_results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2,
            "nonzero": nonzero
        })

best_elastic = min(
    elastic_results,
    key=lambda x: x["RMSE"]
)

best_alpha_elastic = best_elastic["alpha"]
best_l1_ratio = best_elastic["l1_ratio"]

print("Лучшие параметры ElasticNet:")
print("alpha =", best_alpha_elastic)
print("l1_ratio =", best_l1_ratio)
print("RMSE =", best_elastic["RMSE"])
print("R² =", best_elastic["R2"])

Лучшие параметры ElasticNet:
alpha = 0.0001
l1_ratio = 0.1
RMSE = 0.12933373536060713
R² = 0.6526845465388833


### Полиномиальная регрессия

Полиномиальная регрессия позволяет учитывать нелинейные зависимости между признаками и целевой переменной. Для этого исходные признаки преобразуются путём добавления их степеней и произведений.

Например, для двух признаков при степени полинома $d=2$ преобразование имеет вид:

$$
(x_1, x_2) \rightarrow
(x_1, x_2, x_1^2, x_1x_2, x_2^2).
$$

После такого преобразования к полученным признакам применяется обычная линейная регрессия. Таким образом, модель остаётся линейной относительно своих коэффициентов, но может описывать нелинейные зависимости между исходными признаками и целевой переменной.

В работе исследуется полиномиальное преобразование второй степени и сравнивается с линейной моделью первой степени.

In [ ]:
poly_results = []

for degree in [1, 2]:
    poly = PolynomialFeatures(
        degree=degree,
        include_bias=False
    )

    X_train_poly = poly.fit_transform(X_train_pca)
    X_val_poly = poly.transform(X_val_pca)

    model_poly = LinearRegression()
    model_poly.fit(X_train_poly, y_train.values.ravel())

    y_pred_poly = model_poly.predict(X_val_poly)

    mse = mean_squared_error(y_val, y_pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_pred_poly)

    poly_results.append({
        "degree": degree,
        "features": X_train_poly.shape[1],
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    print(f"degree={degree}")
    print(f"Количество признаков: {X_train_poly.shape[1]}")
    print(f"MSE  = {mse:.6f}")
    print(f"RMSE = {rmse:.6f}")
    print(f"R²   = {r2:.6f}")
    print()

degree=1
Количество признаков: 31
MSE  = 0.016727
RMSE = 0.129333
R²   = 0.652689

degree=2
Количество признаков: 527
MSE  = 0.111272
RMSE = 0.333574
R²   = -1.310394



При степени полинома $d=1$ модель использует 31 признак и показывает MSE = 0.016727, RMSE = 0.129333 и $R^2 = 0.652689$.

При увеличении степени до $d=2$ количество признаков возрастает с 31 до 527, однако качество модели существенно ухудшается: MSE увеличивается до 0.111272, RMSE — до 0.333574, а $R^2$ принимает отрицательное значение -1.310394.

Таким образом, добавление полиномиальных признаков второй степени не улучшило модель. Существенное увеличение размерности признакового пространства могло привести к переобучению и существенно ухудшило качество модели на валидационной выборке. В данном случае линейная зависимость оказывается более подходящей, чем полиномиальная модель второй степени.

In [ ]:
results = []

def add_result(name, reduction, params, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    results.append({
        "Признаки": "Числовые признаки",
        "Снижение размерности": reduction,
        "Модель": name,
        "Параметры": params,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2
    })

In [ ]:
linear_pca = LinearRegression()
linear_pca.fit(X_train_pca, y_train.values.ravel())

pred_linear_pca = linear_pca.predict(X_test_pca)

add_result(
    "Linear Regression",
    "PCA (31)",
    "-",
    y_test,
    pred_linear_pca
)

In [ ]:
ridge_final = Ridge(alpha=10)
ridge_final.fit(X_train_pca, y_train.values.ravel())

pred_ridge = ridge_final.predict(X_test_pca)

add_result(
    "Ridge",
    "PCA (31)",
    "alpha=10",
    y_test,
    pred_ridge
)

In [ ]:
lasso_final = Lasso(
    alpha=best_alpha_lasso,
    max_iter=10000
)

lasso_final.fit(X_train_pca, y_train.values.ravel())
pred_lasso = lasso_final.predict(X_test_pca)

add_result(
    "Lasso",
    "PCA (31)",
    f"alpha={best_alpha_lasso}",
    y_test,
    pred_lasso
)

elastic_final = ElasticNet(
    alpha=best_alpha_elastic,
    l1_ratio=best_l1_ratio,
    max_iter=10000
)

elastic_final.fit(
    X_train_pca,
    y_train.values.ravel()
)

pred_elastic = elastic_final.predict(X_test_pca)

add_result(
    "ElasticNet",
    "PCA (31)",
    f"alpha={best_alpha_elastic}, l1_ratio={best_l1_ratio}",
    y_test,
    pred_elastic
)

In [ ]:
poly_final = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_train_poly = poly_final.fit_transform(X_train_pca)
X_test_poly = poly_final.transform(X_test_pca)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train.values.ravel())

pred_poly = poly_model.predict(X_test_poly)

add_result(
    "Polynomial Regression",
    "PCA (31)",
    "degree=2",
    y_test,
    pred_poly
)

In [ ]:
linear_svd = LinearRegression()
linear_svd.fit(X_train_svd, y_train.values.ravel())

pred_svd = linear_svd.predict(X_test_svd)

add_result(
    "Linear Regression",
    "Truncated SVD (31)",
    "-",
    y_test,
    pred_svd
)

In [ ]:
results_df = pd.DataFrame(results)

results_df[["MSE", "RMSE", "R²"]] = (
    results_df[["MSE", "RMSE", "R²"]].round(6)
)

results_df

,Признаки,Снижение размерности,Модель,Параметры,MSE,RMSE,R²
0,Числовые признаки,PCA (31),Linear Regression,-,0.017524,0.132377,0.631797
1,Числовые признаки,PCA (31),Ridge,alpha=10,0.017517,0.132351,0.631945
2,Числовые признаки,PCA (31),Lasso,alpha=0.0001,0.017517,0.132352,0.631936
3,Числовые признаки,PCA (31),ElasticNet,"alpha=0.0001, l1_ratio=0.1",0.017523,0.132374,0.631814
4,Числовые признаки,PCA (31),Polynomial Regression,degree=2,0.213366,0.461915,-3.483166
5,Числовые признаки,Truncated SVD (31),Linear Regression,-,0.017541,0.132442,0.631439


По результатам итоговой оценки на тестовой выборке наилучшее качество показала Ridge-регрессия с параметром $\alpha=10$ после снижения размерности методом PCA. Для неё получены MSE = 0.017517, RMSE = 0.132351 и $R^2 = 0.631945$.

Результаты обычной линейной регрессии, Ridge, Lasso и ElasticNet оказались очень близкими. Это показывает, что в данном случае регуляризация оказывает небольшое влияние на качество модели. Ridge и Lasso дали небольшое улучшение по сравнению с обычной линейной регрессией.

Линейная регрессия после Truncated SVD также показала близкий результат ($R^2 = 0.631439$). Небольшое различие между PCA и Truncated SVD связано с особенностями методов снижения размерности. При этом оба метода сохранили основную часть информации исходного признакового пространства.

Наихудший результат показала полиномиальная регрессия второй степени. После полиномиального преобразования количество признаков увеличилось с 31 до 527, а $R^2$ на тестовой выборке снизился до -3.483166. Отрицательное значение $R^2$ означает, что модель предсказывает хуже простого использования среднего значения целевой переменной. Резкое ухудшение качества может быть связано с чрезмерным увеличением сложности модели и переобучением.

## Вывод

В ходе лабораторной работы были исследованы методы снижения размерности и построены регрессионные модели для прогнозирования целевой переменной `ViolentCrimesPerPop`.

Для снижения размерности использовались методы PCA и Truncated SVD. Исходное пространство из 124 признаков было сокращено до 31 компоненты с сохранением около 90,16 % дисперсии. Метод PCA был дополнительно реализован самостоятельно на основе ковариационной матрицы, собственных значений и собственных векторов. Сравнение с реализацией `sklearn` показало практически совпадающие результаты, что подтверждает корректность собственной реализации.

На полученных представлениях были исследованы линейная регрессия, Ridge, Lasso, ElasticNet и полиномиальная регрессия. Собственная реализация линейной регрессии методом наименьших квадратов также показала практически полное совпадение с `LinearRegression` из библиотеки `sklearn`.

Наилучший результат на тестовой выборке показала Ridge-регрессия после PCA с $\alpha=10$: MSE = 0.017517, RMSE = 0.132351 и $R^2 = 0.631945$. При этом результаты обычной линейной регрессии, Ridge, Lasso и ElasticNet различаются незначительно, поэтому регуляризация в данном случае даёт лишь небольшое улучшение качества.

PCA и Truncated SVD также показали близкие результаты при одинаковом количестве компонент. Полиномиальная регрессия второй степени, напротив, значительно ухудшила качество: увеличение количества признаков с 31 до 527 привело к $R^2 = -3.483166$ на тестовой выборке. Это показывает, что увеличение сложности модели не обязательно приводит к улучшению качества и в данном случае могло привести к переобучению.

Таким образом, наиболее эффективным из рассмотренных вариантов оказалось сочетание PCA и Ridge-регрессии. Снижение размерности позволило существенно сократить число признаков при сохранении большей части информации, а применение умеренной регуляризации обеспечило наилучшее качество среди исследованных моделей.